# PneumoniaMNIST — Classify Chest X-rays

**Your task:** develop a CNN architecture to classify **normal (0)** and **pneumonia (1)** images using [MedMNIST](https://medmnist.com/). This notebook supplies the data import, training, validation and model export. Your best server-evaluated **test accuracy (%)** determines your leaderboard rank.

Use the standard **28×28 grayscale** dataset and the official splits: **4,708 training**, **524 validation**, **624 test**. The notebook trains and validates; the arena evaluates the official test images.

Save a copy to Drive. Select **Runtime → Change runtime type → T4 GPU** if available; CPU also works for this small dataset.

Repository: [jamie-duell/interactive_coding-main](https://github.com/jamie-duell/interactive_coding-main).

Dataset attribution: [MedMNIST v2, Yang et al., Scientific Data (2023)](https://www.nature.com/articles/s41597-022-01721-8), derived from [Kermany et al., Cell (2018)](https://doi.org/10.1016/j.cell.2018.02.010). [PneumoniaMNIST metadata](https://github.com/MedMNIST/MedMNIST/blob/main/medmnist/info.py) lists **CC BY 4.0**. This notebook uses the images as resized by MedMNIST.

In [ ]:
%pip install -q medmnist==3.0.2

In [ ]:
import importlib
import random
from pathlib import Path
import zipfile

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from medmnist import INFO, PneumoniaMNIST

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CLASSES = ("normal", "pneumonia")
info = INFO["pneumoniamnist"]
assert info["MD5"] == "28209eda62fecd6e6a2d98b1501bb15f", "This task uses the standard 28x28 release."
assert info["label"] == {"0": "normal", "1": "pneumonia"}
print("Training device:", device)
print("MedMNIST task:", info["task"], "| Dataset licence:", info["license"])

## Import the official MedMNIST training and validation splits

The images have **one grayscale channel**, not RGB. Convert each image to a tensor, scale pixels to `[0, 1]`, and normalise using mean and standard deviation `(0.5,)`. The exported inference code uses the same transform.

Use **training images for fitting** and **validation images for model selection**. Do not load the test split during development, use test labels, or use pretrained weights. MedMNIST makes test labels public; this classroom exercise relies on those rules.

In [ ]:
Path("data").mkdir(exist_ok=True)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_data = PneumoniaMNIST(split="train", root="data", download=True, transform=transform, size=28)
validation_data = PneumoniaMNIST(split="val", root="data", download=False, transform=transform, size=28)
assert (len(train_data), len(validation_data)) == (4708, 524)
assert train_data[0][0].shape == (1, 28, 28)
BATCH_SIZE = 64
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
validation_loader = DataLoader(validation_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
train_counts = np.bincount(train_data.labels.reshape(-1), minlength=2)
print(f"{len(train_data):,} training images; {len(validation_data):,} validation images")
print("Training counts:", dict(zip(CLASSES, train_counts.tolist())))

## Your architecture — edit this cell

Start with this small working CNN and develop your own convolutional blocks, number of filters, kernel sizes, pooling and classifier. You can explore batch normalisation and dropout.

Keep the class name `CNN`, callable as `CNN()` without arguments. Input shape: `(batch, 1, 28, 28)`. Output shape: `(batch, 2)` **logits** in the order **normal, pneumonia**; cross-entropy applies the training softmax internally.

Put all custom layers and helpers in this file, using PyTorch and standard Python. The server does not install MedMNIST or torchvision in its inference container. Rerun model creation and training after each architecture edit.

**Explain your design:** what changes will you make, and why?

In [ ]:
%%writefile model.py
import torch
from torch import nn


class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Develop your convolutional blocks and classifier.
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(32, 2)

    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))

In [ ]:
# Reload after each architecture edit, then initialise fresh weights.
import model as architecture
importlib.reload(architecture)
model = architecture.CNN().to(device)
trained_architecture = Path("model.py").read_text()
assert model(torch.zeros(2, 1, 28, 28, device=device)).shape == (2, 2)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

## Train and compare validation metrics

Begin with five epochs, then adjust the training settings for promising architectures. Set `USE_CLASS_WEIGHTS = True` to investigate how weighting affects performance on the less frequent class.

**Accuracy** measures the fraction correct. **Balanced accuracy** averages the recall of normal and pneumonia images. **AUROC** measures how well the predicted pneumonia probabilities rank positive versus negative cases. The arena ranks by **test accuracy**, and also reports balanced accuracy and AUROC.

The export uses the checkpoint with the best **validation accuracy**. Rerunning this cell continues training the current model; rerun model creation for a fresh experiment.

In [ ]:
EPOCHS = 5
LEARNING_RATE = 1e-3
USE_CLASS_WEIGHTS = False
class_weights = len(train_data) / (2 * train_counts)
weight = torch.tensor(class_weights, dtype=torch.float32, device=device) if USE_CLASS_WEIGHTS else None
criterion = nn.CrossEntropyLoss(weight=weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
best_accuracy = -1.0
best_state = None
history = []

for epoch in range(EPOCHS):
    model.train()
    train_correct = train_total = 0
    loss_total = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.reshape(-1).long().to(device)  # MedMNIST targets have shape (N, 1).
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        loss_total += loss.item() * len(labels)
        train_correct += (logits.argmax(1) == labels).sum().item()
        train_total += len(labels)

    model.eval()
    val_targets, val_probabilities = [], []
    with torch.inference_mode():
        for images, labels in validation_loader:
            logits = model(images.to(device))
            val_targets.extend(labels.reshape(-1).numpy().tolist())
            val_probabilities.append(logits.softmax(1).cpu().numpy())
    targets = np.asarray(val_targets)
    probabilities = np.concatenate(val_probabilities)
    val_accuracy = float((probabilities.argmax(1) == targets).mean())
    val_balanced = balanced_accuracy_score(targets, probabilities.argmax(1))
    val_auc = roc_auc_score(targets, probabilities[:, 1])
    history.append({"epoch": epoch + 1, "loss": loss_total / train_total,
                    "train_accuracy": train_correct / train_total, "validation_accuracy": val_accuracy,
                    "validation_balanced_accuracy": val_balanced, "validation_auc": val_auc})
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    print(f"Epoch {epoch + 1}/{EPOCHS}: loss {loss_total / train_total:.4f}, "
          f"train {train_correct / train_total:.1%}, validation {val_accuracy:.1%}, "
          f"balanced {val_balanced:.1%}, AUROC {val_auc:.3f}")

assert best_state is not None, "Train for at least one epoch."
model.load_state_dict(best_state)
model.eval()
print(f"Best validation accuracy: {best_accuracy:.1%}")

## Record your experiments

| Architecture / training change | Epochs | Validation accuracy | Balanced accuracy | AUROC | What did you learn? |
|---|---:|---:|---:|---:|---|
| Starter CNN | | | | | |
| Your first change | | | | | |
| Your next change | | | | | |

Which architecture would you submit? How do the three metrics change with class weighting? Explain your choice using the validation results.

## Export and check your model on CPU

The export cell saves your validation-selected weights and checks that the exported model's predictions match the trained model on real validation images. The server receives raw pixels in row-major order, reshapes them to `(N, 1, 28, 28)`, then uses the same normalisation.

If you alter normalisation or the input size, adjust training and inference consistently. This task expects the standard 28×28 images.

In [ ]:
%%writefile submission.py
"""Export with model.py and weights.pt; MedMNIST is needed only during Colab training."""

from pathlib import Path

import numpy as np
import torch
from model import CNN

_ROOT = Path(__file__).resolve().parent
_model = CNN().cpu()
_model.load_state_dict(torch.load(_ROOT / "weights.pt", map_location="cpu", weights_only=True))
_model.eval()


@torch.inference_mode()
def predict(X):
    out = []
    for start in range(0, len(X), 128):
        batch = np.array(X[start : start + 128], dtype=np.float32, copy=True)
        images = torch.from_numpy(batch.reshape(-1, 1, 28, 28)) / 255.0
        images = (images - 0.5) / 0.5
        out.append(torch.softmax(_model(images), dim=1).numpy())
    return np.concatenate(out, axis=0)


In [ ]:
assert best_state is not None, "Run training before export."
assert Path("model.py").read_text() == trained_architecture, (
    "Architecture changed since model creation. Rerun model creation and training."
)
torch.save(best_state, "weights.pt")

import submission
importlib.reload(submission)
sample_ids = list(range(32))
raw_pixels = validation_data.imgs[sample_ids].reshape(-1, 784)
probabilities = submission.predict(raw_pixels)
with torch.inference_mode():
    images = torch.stack([validation_data[i][0] for i in sample_ids]).to(device)
    expected = model(images).softmax(1).cpu().numpy()
assert probabilities.shape == (32, 2)
assert np.isfinite(probabilities).all()
np.testing.assert_allclose(probabilities.sum(1), 1.0, atol=1e-5)
np.testing.assert_allclose(probabilities, expected, atol=1e-5, rtol=1e-4)

with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as bundle:
    for filename in ("submission.py", "model.py", "weights.pt"):
        bundle.write(filename)
print("CPU inference check passed. Upload submission.zip to your arena student page.")
try:
    from google.colab import files
    files.download("submission.zip")
except ImportError:
    print("Outside Colab: download submission.zip from the notebook file browser.")

## Submit

Upload `submission.zip` on the arena student page. The server computes **test accuracy**, **AUROC** and **balanced accuracy** on all 624 official test images. Your best successful test accuracy determines your rank.

Keep this notebook with your architecture, experiment table and explanations. If an upload fails, check the student page's error message; inference uses PyTorch on CPU with no network, MedMNIST or torchvision.